# Analysis Functions for GAD-2 Dataset
This notebook contains all data processing, cleaning, and analysis functions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import r2_score, mean_squared_error, accuracy_score, confusion_matrix, classification_report

## Data Loading Functions

In [2]:
def load_datasets(csv_path, demo_path, gad_path):
    """Load HDP00619 and HDP01233 datasets."""
    df_619 = pd.read_csv("HDP00619.csv", low_memory=False)
    df_1233_demo = pd.read_excel("HDP01233_Demographics.xlsx")
    df_1233_gad = pd.read_excel("HDP01233_Gad2.xlsx")
    return df_619, df_1233_demo, df_1233_gad

## Recoding Functions

In [3]:
def recode_empstat(x):
    """Recode employment status values."""
    if x == 1:
        return 1
    elif x in [3, 4, 5, 6]:
        return 2
    elif x in [2, 7]:
        return 3
    elif x == 8:
        return 4
    else:
        return x


def recode_sex(x):
    """Recode sex values for HDP01233."""
    if x in [1, 2]:
        return x
    elif x in [3, 4]:
        return 0
    else:
        return np.nan


def recode_ethnic(x):
    """Recode ethnicity values for HDP01233."""
    if x == 1:
        return 1
    elif x in [2, 3, 4]:
        return 0
    else:
        return np.nan


def recode_marstat(x):
    """Recode marital status values for HDP01233."""
    if x in [3, 7]:
        return 1
    elif x == 6:
        return 2
    elif x == 2:
        return 3
    elif x in [1, 4]:
        return 4
    elif x == 5:
        return 5
    else:
        return np.nan

## Data Cleaning Functions

In [4]:
def clean_hdp00619(df_619):
    """Clean and process HDP00619 dataset."""
    # Remove hidden spaces
    df_619.columns = df_619.columns.str.strip()
    
    # Standardize column names
    df_619 = df_619.rename(columns={'scr_gender': 'sex'})
    
    # Keep required columns (removed 'height')
    cols_619 = ['scrid', 'age', 'sex', 'ethnic', 'marstat', 'empstat', 'gad701', 'gad702']
    df_619 = df_619[cols_619]
    
    # Merge duplicate scrid rows (keep first non-null)
    df_619 = (
        df_619
        .groupby('scrid', as_index=False)
        .agg(lambda x: x.dropna().iloc[0] if len(x.dropna()) > 0 else np.nan)
    )
    
    # Remove scrid after merge
    df_619 = df_619.drop(columns=['scrid'])
    
    # Drop rows with any NaN
    df_619 = df_619.dropna()
    
    # Convert to integers
    cols_to_int = ['age', 'sex', 'ethnic', 'marstat', 'empstat', 'gad701', 'gad702']
    df_619[cols_to_int] = df_619[cols_to_int].astype(int)
    
    # Add gad_total and higher_anxiety
    df_619["gad_total"] = df_619["gad701"] + df_619["gad702"]
    df_619["higher_anxiety"] = (df_619["gad_total"] >= 6).astype(int)
    
    # Recode employment status
    df_619["empstat"] = df_619["empstat"].apply(recode_empstat)
    
    return df_619

In [5]:
def clean_hdp01233(df_1233_demo, df_1233_gad):
    """Clean and process HDP01233 dataset."""
    # Merge demographics and GAD data
    df_1233 = pd.merge(df_1233_demo, df_1233_gad, on="subject_id", how="inner")
    
    # Rename columns to match HDP00619
    df_1233 = df_1233.rename(columns={
        "gad2feelnervscale": "gad701",
        "gad2notstopwryscale": "gad702",
        "maristat": "marstat"
    })
    
    # Keep required columns (removed 'height')
    cols_keep = ["age", "sex", "visit", "ethnic", "marstat", "empstat", "gad701", "gad702"]
    df_1233 = df_1233[cols_keep]
    
    # Drop rows with visit2
    df_1233 = df_1233[~df_1233["visit"].astype(str).str.contains("2")]
    
    # Drop rows with any NaN
    df_1233 = df_1233.dropna()
    
    # Convert to integers
    cols_to_int = ['age', 'sex', 'ethnic', 'marstat', 'empstat', 'gad701', 'gad702']
    df_1233[cols_to_int] = df_1233[cols_to_int].astype(int)
    
    # Add gad_total and higher_anxiety
    df_1233["gad_total"] = df_1233["gad701"] + df_1233["gad702"]
    df_1233["higher_anxiety"] = (df_1233["gad_total"] >= 6).astype(int)
    df_1233 = df_1233.drop(columns=['visit'])
    
    # Recode categorical variables
    df_1233["sex"] = df_1233["sex"].apply(recode_sex)
    df_1233["ethnic"] = df_1233["ethnic"].apply(recode_ethnic)
    df_1233["marstat"] = df_1233["marstat"].apply(recode_marstat)
    
    return df_1233

## Plotting Functions

In [6]:
def plot_scatter_comparison(df_619, df_1233):
    """Create scatter plot comparing age vs GAD total for both datasets."""
    plt.figure(figsize=(8, 6))
    
    plt.scatter(df_619["age"], df_619["gad_total"], label="HDP00619", alpha=0.6)
    plt.scatter(df_1233["age"], df_1233["gad_total"], label="HDP01233", alpha=0.6)
    
    plt.xlabel("Age")
    plt.ylabel("GAD Total Score")
    plt.title("Age vs GAD Total Score (Two Datasets)")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [7]:
def plot_feature_vs_gad(df, feature_name, feature_label):
    """Plot individual feature vs total GAD score with linear regression line."""
    plt.figure(figsize=(8, 6))
    
    # Scatter plot
    plt.scatter(df[feature_name], df["gad_total"], alpha=0.5, edgecolors='k', linewidths=0.5)
    
    # Fit and plot linear regression line
    X_feat = df[[feature_name]].values
    y_gad = df["gad_total"].values
    
    # Remove any NaN or inf values
    mask = ~(np.isnan(X_feat.flatten()) | np.isnan(y_gad) | 
             np.isinf(X_feat.flatten()) | np.isinf(y_gad))
    X_feat_clean = X_feat[mask].reshape(-1, 1)
    y_gad_clean = y_gad[mask]
    
    if len(X_feat_clean) > 0:
        lr = LinearRegression()
        lr.fit(X_feat_clean, y_gad_clean)
        
        # Create line for plotting
        X_line = np.linspace(X_feat_clean.min(), X_feat_clean.max(), 100).reshape(-1, 1)
        y_line = lr.predict(X_line)
        
        plt.plot(X_line, y_line, 'r-', linewidth=2, 
                label=f'Linear Fit (coef={lr.coef_[0]:.3f})')
        plt.legend()
    
    plt.xlabel(feature_label)
    plt.ylabel("GAD Total Score")
    plt.title(f"{feature_label} vs GAD Total Score")
    plt.tight_layout()
    plt.show()

## Model Preparation Functions

In [8]:
def prepare_model_data(df):
    """Prepare features and targets for modeling."""
    # Removed 'height' from features
    X = df[['age', 'sex', 'marstat', 'empstat', 'ethnic']]
    y_linear = df['gad_total']
    y_logistic = df['higher_anxiety']
    
    # Dummy encode categorical variables
    X = pd.get_dummies(X, drop_first=True)
    
    # Ensure numeric
    X = X.apply(pd.to_numeric)
    
    # Clean data
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(0)
    
    return X, y_linear, y_logistic

## Model Training Functions

In [9]:
def train_linear_regression(X_train, y_train_lin, X_test, y_test_lin):
    """Train and evaluate linear regression model."""
    lin_model = LinearRegression()
    lin_model.fit(X_train, y_train_lin)
    
    y_pred_lin = lin_model.predict(X_test)
    
    results = {
        'model': lin_model,
        'r2': r2_score(y_test_lin, y_pred_lin),
        'rmse': np.sqrt(mean_squared_error(y_test_lin, y_pred_lin)),
        'predictions': y_pred_lin
    }
    
    return results


def train_logistic_regression(X_train, y_train_log, X_test, y_test_log):
    """Train and evaluate logistic regression model."""
    log_model = LogisticRegression(max_iter=1000)
    log_model.fit(X_train, y_train_log)
    
    y_pred_log = log_model.predict(X_test)
    
    results = {
        'model': log_model,
        'accuracy': accuracy_score(y_test_log, y_pred_log),
        'confusion_matrix': confusion_matrix(y_test_log, y_pred_log),
        'classification_report': classification_report(y_test_log, y_pred_log),
        'predictions': y_pred_log
    }
    
    return results


def train_decision_tree(X_train, y_train_log, X_test, y_test_log, X):
    """Train and evaluate decision tree classifier."""
    tree_model = DecisionTreeClassifier(random_state=42)
    tree_model.fit(X_train, y_train_log)
    
    y_pred_tree = tree_model.predict(X_test)
    
    results = {
        'model': tree_model,
        'accuracy': accuracy_score(y_test_log, y_pred_tree),
        'predictions': y_pred_tree,
        'feature_names': X.columns
    }
    
    return results


def train_random_forest(X_train, y_train_log, X_test, y_test_log):
    """Train and evaluate random forest classifier."""
    rf_model = RandomForestClassifier(n_estimators=200, random_state=42)
    rf_model.fit(X_train, y_train_log)
    
    y_pred_rf = rf_model.predict(X_test)
    
    results = {
        'model': rf_model,
        'accuracy': accuracy_score(y_test_log, y_pred_rf),
        'predictions': y_pred_rf
    }
    
    return results

## Visualization Functions for Model Results

In [10]:
def plot_decision_tree(tree_model, feature_names):
    """Visualize decision tree."""
    plt.figure(figsize=(20, 10))
    plot_tree(
        tree_model,
        feature_names=feature_names,
        class_names=["Low Anxiety", "High Anxiety"],
        filled=True
    )
    plt.tight_layout()
    plt.show()


def plot_feature_importance(rf_model, X):
    """Plot random forest feature importance."""
    importance = pd.Series(
        rf_model.feature_importances_,
        index=X.columns
    ).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 6))
    importance.plot(kind="bar")
    plt.title("Random Forest Feature Importance")
    plt.ylabel("Importance")
    plt.xlabel("Features")
    plt.tight_layout()
    plt.show()
    
    return importance


def plot_linear_coefficients(lin_model, X):
    """Plot linear regression coefficients."""
    coefficients = pd.Series(
        lin_model.coef_,
        index=X.columns
    ).sort_values(ascending=False)
    
    plt.figure(figsize=(10, 6))
    coefficients.plot(kind="bar")
    plt.title("Linear Regression Coefficients")
    plt.ylabel("Coefficient Value")
    plt.xlabel("Features")
    plt.axhline(y=0, color='r', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()
    
    return coefficients